# 캐시 라인과 메모리 접근 패턴 - CPU 캐시와 행렬 연산 최적화

- Tutorial ID: `ull-4`
- Tutorial: 캐시 라인과 메모리 접근 패턴
- Section ID: `ull-4-1`
- Section: CPU 캐시와 행렬 연산 최적화

---

## 이 노트북에서 배우는 것

똑같은 행렬 곱셈이라도 **데이터를 어떤 순서로 읽고 쓰는가**에 따라 실행 속도가 수십 배까지 차이 날 수 있습니다.
CPU가 계산하는 속도와, CPU가 메모리에서 데이터를 가져오는 속도는 서로 다르기 때문입니다.
이 노트북은 그 이유를 코드로 직접 실행해 눈으로 확인하기 위한 실습 자료입니다.

배우는 내용:
1. CPU 캐시와 "캐시 라인(cache line)"이 무엇인지
2. 행렬이 메모리에 실제로 어떻게 저장되는지 (row-major order)
3. 똑같은 데이터를 "행 우선"으로 읽을 때와 "열 우선"으로 읽을 때 속도가 왜 다른지
4. 행렬 곱셈을 "타일(tile)" 단위로 쪼개면 왜 더 빨라지는지 (타일링 / 블로킹)
5. GPU의 메모리 구조는 CPU와 무엇이 다른지
6. NumPy의 stride(스트라이드)와 view(뷰)가 메모리를 어떻게 다루는지

### 사전 지식
- Python 기본 문법, NumPy 배열 인덱싱(`arr[i, j]`)에 대한 이해 정도면 충분합니다.
- "캐시", "스트라이드" 같은 용어를 처음 들어도 괜찮습니다 — 코드를 보기 전에 비유와 그림으로 먼저 설명합니다.

### 실습 방법
- 이 노트북은 NumPy와 Matplotlib만 사용하므로 Jupyter, Google Colab, 로컬 환경 어디서든 실행할 수 있습니다.
- 셀을 **위에서부터 순서대로** 실행하세요. 뒤쪽 셀은 앞쪽 셀에서 정의한 함수와 변수를 그대로 이어서 사용합니다.
- 출력되는 ms(밀리초) 숫자 자체는 컴퓨터마다 다르게 나옵니다. 중요한 건 절댓값이 아니라
  **"어떤 방식이 몇 배 더 빠른가"라는 상대적인 경향**입니다. 이 경향은 어떤 컴퓨터에서 실행해도 비슷하게 나타납니다.


In [ ]:
import numpy as np
import time

# 실행 시간을 "숫자"로만 보지 않고 "그림"으로도 비교하기 위해 matplotlib을 사용합니다.
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


def set_korean_font():
    '''matplotlib 그래프에서 한글이 깨지지 않도록, 시스템에 설치된 한글 폰트를 찾아 적용합니다.

    참고: 한글 폰트가 없는 환경(일부 Colab/서버)에서는 그래프의 한글 제목/축 이름이
    빈 사각형(口)으로 보일 수 있습니다. 이것은 코드 오류가 아니라 "폰트가 없다"는
    문제이므로, 실행 결과 자체(출력되는 숫자, 그래프의 막대/선 모양)에는 전혀 영향이
    없습니다. 그래도 한글이 깨지는 게 불편하다면, Colab에서는 아래 두 줄을 실행한 뒤
    런타임을 재시작하면 해결됩니다:
        !apt-get install -y fonts-nanum -qq
        !fc-cache -fv
    '''
    candidates = ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams["font.family"] = name
            plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트 사용 시 마이너스(-) 기호가 깨지는 것을 방지
            print(f"  (그래프용 한글 폰트 '{name}'를 사용합니다.)")
            return
    print("  (시스템에서 한글 폰트를 찾지 못했습니다 - 그래프의 한글 텍스트가 깨져 보일 수 있습니다.")
    print("   위 set_korean_font() 함수의 설명을 참고해서 한글 폰트를 설치해보세요.)")


print("=" * 62)
print("캐시 라인과 메모리 접근 패턴")
print("=" * 62)

set_korean_font()

## 0. 시작하기 전에 — 왜 "어떻게 읽느냐"가 중요할까?

CPU는 1초에 수십억 번 연산을 할 수 있을 정도로 빠릅니다. 하지만 메인 메모리(RAM)에서 데이터를 하나
가져오는 데에는 그보다 훨씬 더 오랜 시간이 걸립니다. 이 속도 차이를 메우기 위해 CPU와 메모리 사이에는
여러 단계의 **캐시(cache)**가 끼어 있습니다.

비유로 생각해보면 이렇습니다.

- **레지스터**는 지금 펼쳐 놓고 보고 있는 **책상 위의 책**입니다. 손만 뻗으면 바로 보입니다.
- **L1 → L2 → L3 캐시**는 점점 멀어지는 **내 방의 책장들**입니다. 책상보다는 멀지만, 의자에서 몇 걸음만
  움직이면 됩니다. 책장이 멀어질수록(L1 → L3) 더 많은 책을 꽂을 수 있지만, 가져오는 데 시간도 더 걸립니다.
- **메인 메모리(RAM, DRAM)**는 **동네 도서관**입니다. 책장에 없는 책을 보려면 도서관까지 다녀와야 하니,
  훨씬 더 오랜 시간이 걸립니다.

CPU가 어떤 데이터를 요청했을 때:
- 그 데이터가 책상이나 책장(레지스터/캐시)에 이미 있으면 → **캐시 히트(cache hit)**. 매우 빠릅니다.
- 책장에도 없어서 도서관(메인 메모리)까지 다녀와야 한다면 → **캐시 미스(cache miss)**. 수십~수백 배
  더 오래 걸립니다.

그래서 똑같은 계산이라도, **데이터를 책장에 꽂힌 순서 그대로 차례차례 읽는지, 아니면 책장 이곳저곳을
무작위로 뒤지는지**에 따라 전체 속도가 크게 달라집니다. 이 노트북에서는 이 차이를 행렬(matrix) 연산으로
직접 확인해봅니다.


## 1. 캐시 라인(Cache Line)이란?

CPU는 메모리에서 데이터를 **1바이트씩** 가져오지 않습니다. 마트에서 물을 살 때 한 병씩 여러 번 왔다
갔다 하기보다 **한 박스(묶음)**로 사 오는 것이 효율적인 것처럼, CPU도 메모리에서 데이터를 **고정된
크기의 묶음** 단위로 가져옵니다. 이 묶음을 **캐시 라인(cache line)**이라고 부르며, 대부분의 현대
CPU에서 그 크기는 **64바이트**입니다.

즉, 배열에서 `arr[0]` 딱 하나만 읽어도 CPU는 사실 그 주변 데이터까지 한꺼번에 캐시로 가져옵니다.
`float32`(4바이트) 값이라면, 캐시 라인 하나에 **64 ÷ 4 = 16개**의 float32 값이 들어갑니다. 즉
`arr[0]`을 읽는 순간 `arr[1]`부터 `arr[15]`까지도 "공짜로" 캐시에 함께 올라와 있을 가능성이 높다는
뜻입니다.

여기서 한 가지 전략이 나옵니다: **이미 캐시에 올라온 이웃 데이터를 최대한 활용하도록 코드를 짜면
더 빠르다.** 아래 코드로 직접 숫자를 확인해봅시다.


In [ ]:
print("\n1. 캐시 라인 기초")
print("-" * 50)

cache_line_bytes = 64       # 대부분의 x86 / ARM CPU에서 흔히 쓰이는 캐시 라인 크기 (bytes)
float32_bytes = 4           # np.float32 값 하나의 크기 (bytes)
floats_per_line = cache_line_bytes // float32_bytes

print(f"  캐시 라인 크기: {cache_line_bytes} bytes")
print(f"  float32 크기: {float32_bytes} bytes")
print(f"  캐시 라인 하나에 들어가는 float32 개수: {floats_per_line}개")
print(f"  -> 즉 float32 값 1개를 읽으면, 바로 옆에 있는 {floats_per_line - 1}개도 같이 캐시에 올라옵니다.")

# -----------------------------------------------------------
# 작은 4x4 배열로 직접 확인해보기
# -----------------------------------------------------------
arr = np.zeros((4, 4), dtype=np.float32)
print(f"\n  예시) 4x4 float32 배열:")
print(f"    배열 전체 크기      : {arr.nbytes} bytes  (4행 x 4열 x 4바이트)")
print(f"    필요한 캐시 라인 수 : {arr.nbytes / cache_line_bytes:.2f}개")
print(f"    한 행(row)의 크기   : {4 * 4} bytes  -> 캐시 라인 1개(64바이트)보다 작음")
print(f"    -> 4x4 배열은 한 행이 캐시 라인 하나에 통째로 들어갑니다!")

# -----------------------------------------------------------
# 행렬 크기가 커지면 한 행이 캐시 라인 여러 개로 나뉩니다.
# -----------------------------------------------------------
print(f"\n  행렬 크기(N)에 따라 '한 행'이 차지하는 캐시 라인 수가 어떻게 늘어나는지 보면:")
for N in [4, 16, 64, 256, 1024]:
    row_bytes = N * float32_bytes
    lines_per_row = row_bytes / cache_line_bytes
    print(f"    N={N:>5} -> 한 행 = {row_bytes:>6} bytes -> 캐시 라인 {lines_per_row:>7.1f}개 필요")


## 2. 행렬은 메모리에 어떻게 저장될까? (Row-major Order)

우리는 행렬을 행과 열이 있는 2차원 표로 생각하지만, 실제 컴퓨터 메모리는 **한 줄로 길게 늘어선
1차원 공간**입니다. 그래서 2차원 배열도 결국 메모리에는 한 줄로 펴서 저장됩니다.

NumPy(그리고 C언어)는 기본적으로 **"한 행을 먼저 다 쓰고, 그 다음 행을 쓰는"** 방식을 사용합니다.
이것을 **row-major order(행 우선 순서)**라고 부릅니다.

예를 들어 3행 4열짜리 배열이 있다면:

```
논리적인 모양 (3행 x 4열):
[[ 0,  1,  2,  3],
 [ 4,  5,  6,  7],
 [ 8,  9, 10, 11]]

메모리에 실제로 저장된 순서 (한 줄로 펴면):
[0, 1, 2, 3 | 4, 5, 6, 7 | 8, 9, 10, 11]
   0행          1행           2행
```

즉, **`arr[0,0]`과 `arr[0,1]`은 메모리상에서 바로 옆자리(이웃)**입니다. 반대로
**`arr[0,0]`과 `arr[1,0]`은 메모리상에서 한 행만큼(이 예시에서는 4칸, 16바이트) 떨어진 곳**에 있습니다.

이 사실을 1번에서 배운 "캐시 라인" 개념과 합쳐보면 중요한 결론이 나옵니다.

- **같은 행 안에서 열 방향으로 이동하며 읽으면** -> 메모리상 이웃을 차례로 읽는 것이므로, 한 번 가져온
  캐시 라인을 계속 재사용할 수 있습니다. (cache-friendly, 캐시 친화적)
- **같은 열 안에서 행 방향으로 이동하며 읽으면** -> 매번 멀리 떨어진 메모리 주소로 "점프"해야 하므로,
  캐시 라인을 거의 매번 새로 가져와야 합니다. (cache-unfriendly, 캐시 비친화적)

아래 코드에서 똑같은 배열을 "행 우선"으로 순회할 때와 "열 우선"으로 순회할 때 시간이 얼마나
차이 나는지 직접 측정해봅니다.


In [ ]:
print("\n2. 행 순회 vs 열 순회 성능")
print("-" * 50)


def row_traverse(arr):
    '''행(row)을 먼저 고정하고, 그 안에서 열(column)을 옆으로 이동하며 순회합니다.
    -> 메모리 주소가 차례로(연속으로) 증가하는 순서로 읽으므로 캐시 친화적입니다.'''
    s = np.float32(0)
    for i in range(arr.shape[0]):        # 바깥 루프: 행(row) 번호
        for j in range(arr.shape[1]):    # 안쪽 루프: 열(column) 번호 -> 한 칸씩 이웃으로 이동
            s += arr[i, j]
    return s


def col_traverse(arr):
    '''열(column)을 먼저 고정하고, 그 안에서 행(row)을 아래로 이동하며 순회합니다.
    -> 매번 한 행(row) 크기만큼 메모리를 건너뛰므로 캐시 비친화적입니다.'''
    s = np.float32(0)
    for j in range(arr.shape[1]):        # 바깥 루프: 열(column) 번호
        for i in range(arr.shape[0]):    # 안쪽 루프: 행(row) 번호 -> 매번 멀리 점프
            s += arr[i, j]
    return s


n = 200
A = np.random.randn(n, n).astype(np.float32)
repeats = 3

start = time.perf_counter()
for _ in range(repeats):
    row_traverse(A)
row_time = (time.perf_counter() - start) / repeats

start = time.perf_counter()
for _ in range(repeats):
    col_traverse(A)
col_time = (time.perf_counter() - start) / repeats

print(f"  {n}x{n} 배열 순회 (총 {n*n:,}개 원소를 더함):")
print(f"    행 우선 순회: {row_time*1000:.1f} ms")
print(f"    열 우선 순회: {col_time*1000:.1f} ms")
print(f"    -> 열 우선 순회가 {col_time/row_time:.1f}배 느림")
print(f"\n  (이 비율은 컴퓨터, 캐시 크기, OS 상태에 따라 크게 달라질 수 있습니다.")
print(f"   1.0배에 가깝게 나오거나, 심지어 거꾸로 나오는 경우도 있습니다 - 그 이유는 바로 다음 설명에서 다룹니다.)")

# -----------------------------------------------------------
# 시각화: 막대그래프로 두 방식의 속도 차이를 한눈에 보기
# -----------------------------------------------------------
plt.figure(figsize=(5, 4))
times_ms = [row_time * 1000, col_time * 1000]
bars = plt.bar(
    ["행 우선 순회\n(cache-friendly)", "열 우선 순회\n(cache-unfriendly)"],
    times_ms,
    color=["#4C72B0", "#DD8452"],
)
plt.ylabel("실행 시간 (ms)")
plt.title(f"{n}x{n} 배열 순회 시간 비교")
for bar, t in zip(bars, times_ms):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{t:.1f}ms",
              ha="center", va="bottom")
plt.tight_layout()
plt.show()


### 한 가지 함정 — 파이썬 반복문 자체의 비용

위 결과가 기대한 만큼 차이가 크지 않거나, 컴퓨터에 따라 거의 1배(즉, 차이가 거의 없음)로 나왔다면
당황할 수 있습니다. 이상한 것이 아닙니다 — 오히려 **중요한 함정 하나**를 보여주는 것입니다.

`arr[i, j]`처럼 원소 하나를 파이썬 반복문 안에서 꺼낼 때마다, 파이썬과 NumPy는 그 한 번의 호출을
위해 자체적으로 적지 않은 "사무 처리"(타입 확인, 객체 생성 등)를 합니다. 이 호출 자체의 비용이
보통 수십~수백 나노초 수준인데, 우리가 측정하려는 "캐시 친화적 vs 비친화적"의 차이도 비슷한
수준(수~수십 나노초)일 수 있습니다. 즉, **잴 대상보다 측정 도구 자체의 무게가 더 무거운 상황**이
벌어질 수 있다는 뜻입니다. 마치 손톱 끝의 미세한 길이를 재겠다고 손에 두꺼운 장갑을 낀 채로 자를
대는 셈입니다.

이 문제를 피하려면, **파이썬 반복문 호출 횟수 자체를 줄이고, 한 번 호출할 때 더 많은 일을
시키면** 됩니다. 마침 NumPy에는 한 행 전체, 또는 한 열 전체를 한 번에 더하는 `.sum()`이 있습니다.

- `arr[i, :].sum()` → i번째 행 전체(메모리상 연속된 데이터)를 한 번의 호출로 더함
- `arr[:, j].sum()` → j번째 열 전체(메모리상 듬성듬성 떨어진 데이터)를 한 번의 호출로 더함

이렇게 하면 파이썬 반복문은 "행(또는 열)의 개수"만큼만 돌고, 실제 원소 하나하나를 더하는 작업은
NumPy 내부의 빠른 C 코드가 처리합니다. 그 결과 파이썬 호출 비용이 거의 사라지고, **메모리를
연속으로 읽는지(행) 아니면 듬성듬성 읽는지(열)에 따른 진짜 차이**가 훨씬 선명하게 드러납니다.


In [ ]:
print("\n  더 정확한 측정: 행/열을 한 번에 sum()하기")
print("-" * 50)


def row_sum_traverse(arr):
    '''각 행(row) 전체를 한 번의 NumPy 호출로 합산합니다. (메모리상 연속 구간을 읽음)'''
    total = np.float32(0)
    for i in range(arr.shape[0]):
        total += arr[i, :].sum()   # i행 전체 = 메모리에서 연속된 한 덩어리
    return total


def col_sum_traverse(arr):
    '''각 열(column) 전체를 한 번의 NumPy 호출로 합산합니다. (메모리상 듬성듬성 떨어진 위치를 읽음)'''
    total = np.float32(0)
    for j in range(arr.shape[1]):
        total += arr[:, j].sum()   # j열 전체 = 매 원소가 한 행(row) 크기만큼씩 떨어져 있음
    return total


sizes = [200, 500, 1000, 2000, 4000]
row_times, col_times, ratios = [], [], []

print("  (배열이 클수록 시간이 더 걸리므로, 이 셀은 몇 초 정도 걸릴 수 있습니다.)")
for size in sizes:
    test_arr = np.random.randn(size, size).astype(np.float32)

    row_sum_traverse(test_arr)  # 첫 호출은 워밍업 목적으로 한 번 버립니다.
    col_sum_traverse(test_arr)

    start = time.perf_counter()
    row_sum_traverse(test_arr)
    t_row = time.perf_counter() - start

    start = time.perf_counter()
    col_sum_traverse(test_arr)
    t_col = time.perf_counter() - start

    row_times.append(t_row * 1000)
    col_times.append(t_col * 1000)
    ratio = t_col / t_row
    ratios.append(ratio)
    print(f"  N={size:>4} : 행 합={t_row*1000:8.3f}ms  열 합={t_col*1000:8.3f}ms  비율={ratio:.2f}x")

print("\n  -> 배열이 작을 때(예: N=200)는 데이터가 캐시 안에 거의 다 들어가서 차이가 작습니다.")
print("     배열이 캐시보다 커질수록(N이 커질수록), 열 방향 합산은 매번 캐시 미스를 일으키기")
print("     쉬워져서 행 방향 합산보다 훨씬(N=4000에서는 10배 가까이) 느려집니다.")

# -----------------------------------------------------------
# 시각화 1: 배열 크기별 비율 - 캐시보다 커질수록 격차가 벌어지는 모습
# -----------------------------------------------------------
plt.figure(figsize=(5, 4))
plt.plot(sizes, ratios, marker="o", color="#C44E52")
plt.xlabel("배열 한 변의 크기 (N, N x N 배열)")
plt.ylabel("열 합산 시간 / 행 합산 시간 (배)")
plt.title("배열이 커질수록 캐시 비친화적 접근이\n더 느려지는 정도")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# -----------------------------------------------------------
# 시각화 2: 크기별 실제 실행 시간 (로그 스케일) - 두 선이 갈라지는 모습
# -----------------------------------------------------------
plt.figure(figsize=(5, 4))
plt.plot(sizes, row_times, marker="o", label="행 우선 (연속 메모리)", color="#4C72B0")
plt.plot(sizes, col_times, marker="o", label="열 우선 (듬성듬성 메모리)", color="#DD8452")
plt.xlabel("배열 한 변의 크기 (N)")
plt.ylabel("실행 시간 (ms, 로그 스케일)")
plt.yscale("log")
plt.title("배열 크기별 행 합산 vs 열 합산 시간")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()


## 3. 행렬 곱 타일링(Tiling / Blocking) 최적화

행렬 곱셈 `C = A @ B`를 가장 단순하게 구현하면 3중 for문이 됩니다.

```
for i in range(M):
    for j in range(N):
        for k in range(K):
            C[i, j] += A[i, k] * B[k, j]
```

문제는 안쪽 `k` 루프가 돌 때 `B[k, j]`를 읽는 방식이, 2번에서 본 **열 우선 접근**과 똑같다는
점입니다 (`k`가 바뀔 때마다 행이 바뀌므로). 행렬이 커질수록 `A`의 한 행과 `B`의 한 열을 전부
캐시에 올려 두기가 어려워지고, 캐시 미스가 점점 늘어납니다.

**타일링(tiling, 또는 블로킹 blocking)**은 이 문제를 줄이는 전통적인 기법입니다. 아이디어는
간단합니다.

> 행렬 전체를 한 번에 계산하지 말고, **작은 정사각형 조각(타일, tile)** 단위로 잘라서, 한 타일
> 안에서 계산을 모두 끝낸 뒤 다음 타일로 넘어가자.

비유로 설명하면: 책장 전체를 정리할 때 책장 전체를 한꺼번에 들고 보지 않고, **한 칸씩** 꺼내서
책상 위에서 정리한 다음 다시 꽂는 것과 비슷합니다. 한 번에 다루는 양(타일 크기)이 책상(캐시)
크기에 맞으면, 책장과 책상을 왔다 갔다 하는 횟수가 크게 줄어듭니다.

타일 하나가 캐시(특히 가장 빠르고 작은 L1 캐시, 보통 32KB)에 들어갈 만큼 작으면, 그 타일에
속한 데이터를 여러 번 재사용해도 매번 메인 메모리까지 갈 필요가 없습니다. 이것이 타일링이
빨라지는 핵심 이유입니다.


In [ ]:
print("\n3. 행렬 곱 타일링 최적화")
print("-" * 50)


def naive_matmul(A, B):
    '''순진한(naive) 3중 루프 행렬 곱. 최적화 없이 수학적 정의 그대로 계산합니다.'''
    M, K = A.shape
    K2, N = B.shape
    C = np.zeros((M, N), dtype=A.dtype)
    for i in range(M):          # 결과 C의 행
        for j in range(N):      # 결과 C의 열
            for k in range(K):  # A의 i행과 B의 j열을 내적(dot product)하기 위한 합산 인덱스
                C[i, j] += A[i, k] * B[k, j]
    return C


def tiled_matmul(A, B, tile_size=16):
    '''타일링(블로킹)된 행렬 곱.
    M x N x K 전체를 한 번에 훑지 않고, tile_size 크기의 작은 블록 단위로 나눠서 계산합니다.'''
    M, K = A.shape
    K2, N = B.shape
    C = np.zeros((M, N), dtype=A.dtype)

    # 바깥 3중 루프: "타일" 단위로 이동합니다 (ii, jj, kk = 각 타일의 시작 좌표)
    for ii in range(0, M, tile_size):
        for jj in range(0, N, tile_size):
            for kk in range(0, K, tile_size):
                # 행렬 크기가 tile_size로 딱 나누어지지 않을 수도 있으므로,
                # 타일의 끝 경계가 행렬 범위를 넘지 않도록 min으로 잘라줍니다.
                i_end = min(ii + tile_size, M)
                j_end = min(jj + tile_size, N)
                k_end = min(kk + tile_size, K)

                # 안쪽 3중 루프: 이 작은 타일 안에서만 곱셈/합산을 수행합니다.
                # -> 이 타일에 속한 A, B, C 조각은 충분히 작아서 캐시(특히 L1)에 들어갑니다.
                for i in range(ii, i_end):
                    for j in range(jj, j_end):
                        for k in range(kk, k_end):
                            C[i, j] += A[i, k] * B[k, j]
    return C


n = 64  # 순진한 3중 루프는 매우 느리므로, 시연을 위해 작은 크기를 사용합니다.

A = np.random.randn(n, n).astype(np.float32)
B = np.random.randn(n, n).astype(np.float32)

start = time.perf_counter()
C_naive = naive_matmul(A, B)
t_naive = time.perf_counter() - start

start = time.perf_counter()
C_tiled = tiled_matmul(A, B, tile_size=16)
t_tiled = time.perf_counter() - start

start = time.perf_counter()
C_numpy = A @ B   # NumPy 내부적으로 BLAS라는, 수십 년간 고도로 최적화된 라이브러리를 사용합니다.
t_numpy = time.perf_counter() - start

# 최적화는 "속도"만 바꾸고 "정답"은 바꾸면 안 됩니다 - 먼저 결과 값이 일치하는지 확인합니다.
print(f"  결과 값 검증 (NumPy 결과와 같은가?):")
print(f"    naive  == numpy ? {np.allclose(C_naive, C_numpy, atol=1e-3)}")
print(f"    tiled  == numpy ? {np.allclose(C_tiled, C_numpy, atol=1e-3)}")

print(f"\n  {n}x{n} 행렬 곱 실행 시간:")
print(f"    순진한 3중 루프:  {t_naive*1000:8.1f} ms")
print(f"    타일링 (16x16):   {t_tiled*1000:8.1f} ms")
print(f"    NumPy (BLAS):     {t_numpy*1000:8.3f} ms")

if t_tiled < t_naive:
    print(f"\n  이번 실행에서는 타일링이 순진한 방식보다 {t_naive/t_tiled:.2f}배 빠르게 나왔습니다.")
else:
    print(f"\n  이번 실행에서는 타일링이 더 빨라지지 않았습니다 (오히려 {t_tiled/t_naive:.2f}배 느림).")
    print(f"  당황하지 마세요 - 2번 섹션의 '파이썬 반복문 자체의 비용'과 똑같은 이유입니다.")
    print(f"  `C[i, j] += A[i, k] * B[k, j]` 한 번을 실행할 때마다 파이썬/NumPy 호출 비용이")
    print(f"  들어가는데, 이 비용이 보통 '캐시에서 가져오는지 메인 메모리에서 가져오는지'의")
    print(f"  차이보다 더 큽니다. 즉 타일링이라는 아이디어 자체는 옳지만, 순수 파이썬")
    print(f"  반복문 수준에서는 그 효과가 가려질 수 있습니다 - 컴파일된 코드(C/C++/어셈블리)에서야")
    print(f"  제대로 드러납니다.")

print(f"\n  반면 NumPy(BLAS)는 순진한 방식보다 {t_naive/t_numpy:.0f}배 빠릅니다.")
print(f"  -> BLAS는 우리가 흉내 낸 타일링뿐 아니라, SIMD(벡터 연산), 멀티스레딩, 어셈블리 수준")
print(f"     최적화까지 '미리 컴파일된 코드'로 구현해 둔 덕분에 이런 속도를 냅니다.")
print(f"     타일링 자체는 BLAS 내부에서도 실제로 쓰이는 핵심 기법이지만, 그 효과를 보려면")
print(f"     파이썬 반복문이 아니라 컴파일되는 언어로 구현해야 한다는 것이 핵심 교훈입니다.")

print(f"\n  타일 크기(16x16)와 캐시 용량 비교:")
tile_bytes = 16 * 16 * 4
print(f"    16x16 타일 1개 크기 = 16 x 16 x 4 bytes = {tile_bytes:,} bytes")
print(f"    L1 캐시 용량(보통 32KB = 32,768 bytes)에는 약 {32768 // tile_bytes}개의 타일을 동시에 올릴 수 있습니다.")
print(f"    -> A, B, C 각각의 타일을 모두 캐시에 올려두고 재사용할 수 있는 여유가 충분합니다.")


### 타일 크기는 아무거나 괜찮을까?

이론적으로는, 타일이 너무 작으면 반복문 자체를 도는 오버헤드가 늘어나고, 너무 크면 캐시에 다
들어가지 못해서 다시 캐시 미스가 늘어납니다. 즉, "적당한" 타일 크기가 존재합니다.

단, 앞에서 설명한 것처럼 순수 파이썬 반복문에서는 호출 비용이 타일링 효과를 가릴 수 있어서,
타일 크기에 따른 차이가 그래프에 명확하게 나타나지 않을 수도 있습니다. 그럼에도 불구하고 이
실험을 해보는 이유는, **"타일 크기가 다르면 실행 시간이 다를 수 있다"는 개념 자체**를 손으로
직접 확인하기 위해서입니다.

실제 라이브러리(BLAS, cuBLAS)의 컴파일된 코드에서는 타일 크기에 따른 차이가 매우 선명하게
나타나며, 이 최적 타일 크기를 자동으로(또는 컴파일 시점에) 결정하는 기법이 곧 고성능 행렬
연산 라이브러리의 핵심 노하우입니다.


In [ ]:
print("\n  타일 크기별 실행 시간 비교")
print("-" * 50)

tile_sizes = [4, 8, 16, 32, 64]   # 64는 행렬 전체 크기와 같아서, 결국 naive_matmul과 똑같아집니다.
tile_times = []

for ts in tile_sizes:
    start = time.perf_counter()
    tiled_matmul(A, B, tile_size=ts)
    t = time.perf_counter() - start
    tile_times.append(t)
    print(f"  tile_size={ts:>3} : {t*1000:8.2f} ms")

plt.figure(figsize=(5, 4))
plt.plot(tile_sizes, [t * 1000 for t in tile_times], marker="o", color="#55A868")
plt.xlabel("타일 크기 (tile_size)")
plt.ylabel("실행 시간 (ms)")
plt.title(f"{n}x{n} 행렬 곱: 타일 크기에 따른 속도")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n  -> 파이썬 반복문에서는 타일 크기에 따른 차이가 작게(또는 거꾸로) 나올 수 있습니다.")
print("     이것은 반복문 호출 자체의 오버헤드 때문이지, 타일링 아이디어가 틀려서가 아닙니다.")
print("     중요한 것은 tile_size=4(매우 잘게 나눔)일 때 반복문 횟수 자체가 늘어나면서 시간이")
print("     증가하고, tile_size=64(행렬 전체를 한 타일로)일 때는 naive_matmul과 동일해진다는 것입니다.")
print("     실제 컴파일된 코드(C/CUDA)에서는 타일 크기에 따라 수십 % 이상의 속도 차이가 납니다.")


## 4. GPU의 메모리 계층은 CPU와 무엇이 다를까?

지금까지는 CPU 캐시 이야기였습니다. 그런데 딥러닝 모델은 대부분 GPU에서 학습/추론됩니다. GPU도
"가까운 메모리는 빠르고 작다, 먼 메모리는 느리고 크다"는 같은 원리를 따르지만, 구체적인 구조는
CPU와 다릅니다.

먼저 용어를 간단히 정리하면:

- **SM (Streaming Multiprocessor)**: GPU 안에 있는 "코어 묶음" 단위입니다. CPU의 코어와 비슷한
  역할을 하며, GPU 한 개에는 SM이 수십~수백 개 들어 있습니다.
- **공유 메모리 (Shared Memory)**: 각 SM 내부에 있는, 프로그래머가 직접 관리할 수 있는 빠른
  메모리입니다. CPU의 캐시는 보통 하드웨어가 자동으로 관리하지만, GPU의 공유 메모리는 코드에서
  "이 데이터는 여기 저장해 둬"라고 직접 지정할 수 있습니다.
- **HBM (High Bandwidth Memory)**: GPU의 메인 메모리(예: A100의 80GB 메모리)입니다. CPU의
  RAM과 비슷한 역할이지만 훨씬 더 높은 대역폭을 가집니다.

아래 표로 CPU와 GPU(A100 기준)의 메모리 계층을 나란히 비교해봅니다. (수치는 세대/제품에 따라
달라질 수 있는 대략적인 값입니다.)


In [ ]:
print("\n4. GPU vs CPU 메모리 계층")
print("-" * 50)

memory_table_lines = [
    "",
    "  CPU:                            GPU (A100 기준):",
    "  --------------                  --------------------",
    "  레지스터    ~1ns    ~1KB        레지스터     ~1ns    SM당 ~256KB (전체로는 수십MB)",
    "  L1 캐시     ~1ns    32KB        공유 메모리  ~5ns    SM당 192KB (프로그래머가 직접 관리)",
    "  L2 캐시     ~4ns    256KB       L2 캐시      ~20ns   40MB (GPU 전체가 공유)",
    "  L3 캐시     ~12ns   32MB        HBM          ~200ns  80GB (메인 메모리)",
    "  DRAM(RAM)   ~60ns   수~수백GB",
    "",
]
print("\n".join(memory_table_lines))

print("  핵심 차이:")
print("    1) GPU는 코어(SM) 수가 매우 많아서, 동시에 처리할 수 있는 작업(스레드) 수가 CPU보다 훨씬 많습니다.")
print("    2) CPU의 L1/L2/L3 캐시는 하드웨어가 '알아서' 무엇을 캐싱할지 관리합니다 (자동 관리).")
print("    3) GPU의 '공유 메모리'는 프로그래머가 코드로 직접 '이 데이터는 여기 둬라'고 지정합니다 (수동 관리).")
print("    4) 이 수동 관리 덕분에, FlashAttention 같은 기법은 어텐션 연산의 중간 결과를")
print("       느린 HBM에 왕복시키지 않고, 빠른 공유 메모리 안에서 최대한 끝내도록 설계되어 있습니다.")
print("       -> 이것이 FlashAttention이 '같은 연산'을 하면서도 훨씬 빠른 핵심 이유 중 하나입니다.")


## 5. NumPy의 스트라이드(Strides)와 뷰(View)

2번 섹션에서 "row-major 순서로 저장된다"고 배웠습니다. NumPy는 이 사실을 `strides`라는 속성으로
명시적으로 보여줍니다.

**스트라이드(stride)**란, 어떤 차원에서 인덱스를 1 증가시킬 때 메모리 주소가 몇 바이트(byte)
이동해야 하는지를 나타내는 값입니다.

예를 들어 3행 4열, `float32`(4바이트) 배열이 있다면:
- 열 방향으로 한 칸 이동(`j -> j+1`)할 때는 메모리에서 **4바이트**만 이동하면 됩니다 (바로 다음
  칸이 이웃이므로).
- 행 방향으로 한 칸 이동(`i -> i+1`)할 때는 한 행 전체(4개 원소 = 16바이트)를 건너뛰어야 합니다.

즉 이 배열의 `strides`는 `(16, 4)`가 됩니다 — 순서는 (행 방향으로 한 칸 이동할 때의 바이트,
열 방향으로 한 칸 이동할 때의 바이트)입니다.

이걸 알면 임의의 원소 `arr[i, j]`의 메모리 주소를 직접 계산할 수도 있습니다.

```
주소(arr[i, j]) = 시작 주소 + i * strides[0] + j * strides[1]
```

또한 strides 덕분에 NumPy는 아주 똑똑한 트릭을 부립니다. **전치(transpose, `.T`)를 해도 실제
데이터를 복사하지 않고, strides의 순서만 바꿔서** "다른 모양처럼 보이게" 만듭니다. 이렇게 원본과
메모리를 공유하는 배열을 **뷰(view)**라고 부릅니다.

문제는, 전치된 배열의 strides는 원본과 순서가 반대이기 때문에, 전치된 배열을 겉보기엔 평범하게
(행 우선으로) 순회해도 실제로는 2번 섹션에서 본 "열 우선 순회"와 같은 효과가 나서 캐시 비친화적이
된다는 점입니다. 아래 코드로 확인해봅시다.


In [ ]:
print("\n5. 스트라이드와 메모리 뷰")
print("-" * 50)

arr = np.arange(12, dtype=np.float32).reshape(3, 4)
print(f"  원본 배열 (3행 x 4열, C-contiguous):")
print(f"    배열:\n{arr}")
print(f"    strides: {arr.strides} bytes  -> (행 이동 시 {arr.strides[0]}바이트, 열 이동 시 {arr.strides[1]}바이트)")
print(f"    메모리에 저장된 실제 순서 (ravel): {arr.ravel()}")

# -----------------------------------------------------------
# 주소 계산을 직접 해보기: arr[1, 2]의 위치를 strides로 계산
# -----------------------------------------------------------
i, j = 1, 2
offset_bytes = i * arr.strides[0] + j * arr.strides[1]
flat_index = offset_bytes // arr.itemsize
print(f"\n  예시) arr[{i}, {j}]의 위치를 strides로 직접 계산하면:")
print(f"    오프셋 = {i} * {arr.strides[0]} + {j} * {arr.strides[1]} = {offset_bytes} bytes")
print(f"    -> 1차원으로 펼친 배열에서 {flat_index}번째 위치 -> 값 = {arr.ravel()[flat_index]}")
print(f"    실제 arr[{i}, {j}] = {arr[i, j]}  (직접 계산한 값과 일치하는지 확인!)")

# -----------------------------------------------------------
# 전치 = 데이터 복사 없이 strides만 바꿔서 "보이는 모양"만 바꿈
# -----------------------------------------------------------
arr_T = arr.T
print(f"\n  전치 arr.T (4행 x 3열, Non-contiguous):")
print(f"    배열:\n{arr_T}")
print(f"    strides: {arr_T.strides} bytes  (원본과 순서가 뒤바뀐 것을 확인하세요)")
print(f"    원본과 같은 메모리를 공유하는가? {np.shares_memory(arr, arr_T)}")
print(f"    C-contiguous(메모리에 순서대로 저장됨)인가? {arr_T.flags['C_CONTIGUOUS']}")
print(f"    -> 데이터는 그대로인데 strides만 바뀌어서 '전치된 것처럼' 보이는 것입니다.")
print(f"       복사가 일어나지 않으므로 매우 빠릅니다 (거의 공짜 연산)!")

# -----------------------------------------------------------
# 강제로 연속(contiguous)된 메모리로 복사하면
# -----------------------------------------------------------
arr_T_contig = np.ascontiguousarray(arr_T)
print(f"\n  np.ascontiguousarray(arr_T) 적용 후:")
print(f"    strides: {arr_T_contig.strides} bytes")
print(f"    원본과 같은 메모리를 공유하는가? {np.shares_memory(arr, arr_T_contig)}")
print(f"    -> 이번엔 실제로 데이터를 새 메모리에 행 우선 순서로 복사했습니다.")
print(f"       (메모리를 추가로 사용하고, 복사하는 시간도 더 걸립니다.)")
print(f"    -> 만약 전치된 배열을 여러 번 반복해서 행 우선으로 순회해야 한다면,")
print(f"       차라리 한 번 복사해서 연속 메모리로 만들어 두는 게 전체적으로는 더 빠를 수도 있습니다.")


## 정리 및 더 해볼 것들

이 노트북에서 확인한 핵심 내용을 정리하면:

1. **캐시 라인**: CPU는 메모리를 64바이트 단위로 가져옵니다. 한 값을 읽으면 이웃 값도 같이
   캐시에 올라옵니다.
2. **Row-major order**: NumPy(와 C)는 2차원 배열을 "행을 먼저 다 쓰고 다음 행"의 순서로
   메모리에 저장합니다.
3. **행 우선 순회가 (이론적으로) 빠른 이유**: 메모리상 이웃을 순서대로 읽기 때문에 캐시
   라인을 최대한 재사용합니다. 열 우선 순회는 매번 멀리 점프하므로 캐시 미스가 늘어납니다.
4. **측정의 함정**: 파이썬 반복문에서 원소 하나씩 접근하면, 파이썬/NumPy 호출 자체의
   오버헤드가 캐시 효과보다 더 크게 측정 결과를 좌우할 수 있습니다. 한 번의 호출로 더 많은
   일을 시키는(예: 행/열 전체를 `.sum()`) 방식으로 측정해야 진짜 효과가 선명하게 드러납니다.
5. **타일링(블로킹)**: 큰 행렬 연산을 캐시에 들어갈 만한 작은 블록 단위로 나누면, 메모리
   왕복 횟수를 줄여서 빠르게 만들 수 있습니다. 단, 타일 크기는 너무 작거나 크면 오히려
   손해입니다.
6. **GPU 메모리 계층**: CPU와 비슷하게 "가까울수록 빠르고 작다"는 원리를 따르지만, 공유
   메모리를 프로그래머가 직접 관리할 수 있다는 점이 다릅니다. FlashAttention 같은 기법이
   이 특성을 적극 활용합니다.
7. **strides와 view**: 전치(`.T`) 같은 연산은 strides만 바꿔서 데이터 복사 없이 "다르게
   보이게" 만듭니다. 다만 그 결과를 비효율적인 순서로 순회하면 다시 느려질 수 있습니다.

### 더 해보면 좋은 것들 (연습 문제)

- [ ] 2번 섹션의 `row_sum_traverse`/`col_sum_traverse`에 사용한 `sizes` 리스트에 8000,
      16000처럼 더 큰 값을 추가해서 비율이 계속 커지는지, 아니면 어느 순간부터 한계에
      도달하는지 확인해보세요. (단, 메모리 사용량이 커지니 너무 큰 값은 주의하세요.)
- [ ] 3번 섹션의 `tile_size`를 2, 128처럼 극단적인 값으로 바꿔서 실행 시간이 어떻게
      변하는지 확인해보세요.
- [ ] 3차원 배열(`np.random.randn(n, n, n)`)에서도 비슷한 row-major / 순회 순서 효과가
      나타나는지 직접 실험해보세요.
- [ ] `arr.flags`, `arr.itemsize`, `arr.flags['F_CONTIGUOUS']` 같은 속성을 출력해보고
      각각이 무엇을 의미하는지 찾아보세요.
- [ ] (도전) `np.asfortranarray()`로 column-major(Fortran 순서) 배열을 만든 뒤, 이번에는
      "열 우선 순회"가 오히려 더 빨라지는지 확인해보세요.
